# Day 3 Mini Capstone – AI Assistant (APIs + Embeddings + RAG)

**Objective:** Design and present a working AI assistant that combines (1) an LLM API call (Day 1), (2) embeddings and vector search (Day 2), and (3) a RAG / document-aware Q&A chain (Day 3).

**Duration:** 60–90 minutes for design + build; 5–10 minutes per team for presentation.

**Deliverable:** A runnable notebook that implements a minimal end-to-end pipeline, plus a short presentation covering architecture, demo, and one responsible-AI consideration.

## Architecture diagram

Your design should include at least one component from each day.

```
                         ┌──────────────────────────────────────────┐
                         │           AI  ASSISTANT                  │
                         │                                          │
User ─── question ──────▶│   ┌──────────────────────────────────┐   │
                         │   │         Orchestrator              │   │
                         │   └─────┬───────────┬────────────┬────┘   │
                         │         │           │            │        │
                         │         ▼           ▼            ▼        │
                         │  ┌────────────┐ ┌──────────┐ ┌─────────┐ │
                         │  │  Day 1     │ │  Day 2   │ │ Day 3   │ │
                         │  │  LLM API   │ │  Vector  │ │ RAG     │ │
                         │  │  call      │ │  store & │ │ chain   │ │
                         │  │ (generate) │ │  embed   │ │ (retr + │ │
                         │  │            │ │ (search) │ │ gen)    │ │
                         │  └────────────┘ └──────────┘ └─────────┘ │
                         │         │           │            │        │
                         │         └───────────┴────────────┘        │
                         │                     │                     │
                         │                     ▼                     │
User ◀─── answer ───────│           answer + sources                │
         + sources       └──────────────────────────────────────────┘
```

| Layer | Day | What it provides |
|---|---|---|
| LLM API call | Day 1 | Text generation via OpenAI (or another API) |
| Embeddings + vector store | Day 2 | Semantic search over a document collection |
| RAG chain | Day 3 | Retrieve relevant context → feed to LLM → grounded answer |

## Scope and constraints

**Must include:**
- At least one REST / LLM API call (e.g., `ChatOpenAI`)
- Embeddings + vector store (e.g., `HuggingFaceEmbeddings` + Chroma)
- RAG-style answer: retrieve context → prompt → LLM → answer with source attribution

**Optional extensions:**
- Tools (calculator, web search)
- Multi-turn conversation memory
- Evaluation (Recall@k, LLM-as-judge)
- Guardrails (PII filter, content safety)
- Custom UI (Gradio, Streamlit)

**Constraints:**
- Small document set (5–20 docs) — no large-scale deployment
- Keep prompts and chains simple and readable
- Presentation: diagram + live demo preferred over slides

## Design checklist

Use this checklist to plan your assistant before writing code.

| # | Step | Your notes |
|---|---|---|
| 1 | **Choose use case** — e.g., internal KB, customer support draft, compliance Q&A, onboarding guide | |
| 2 | **List APIs** — which LLM model? Any external APIs? | |
| 3 | **Document source** — what documents? how many? how will you chunk them? | |
| 4 | **Sketch the flow** — user question → retrieve → prompt → LLM → answer | |
| 5 | **Responsible AI** — cite sources? filter PII? confidence threshold? | |

## Setup

In [ ]:
import json, os, time, warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

API_KEY = os.environ.get("OPENAI_API_KEY", "")

with open("config.json") as f:
    CONFIG = json.load(f)

print(f"LLM model      : {CONFIG['llm_model']}")
print(f"Embedding model : {CONFIG['embedding_model']}")
print(f"API key set     : {bool(API_KEY)}")

if not API_KEY:
    print("\nNo OPENAI_API_KEY set — simulated responses will be used.")

## Section A: Load documents and build vector store (Day 2)

*This implements the **Day 2** component — embeddings and vector search.*

Replace the sample documents below with your own use-case data.

In [ ]:
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# ── YOUR DOCUMENTS (replace or extend with your own) ──
documents = [
    Document(
        page_content=(
            "Remote work is available to all full-time employees after 90 days. "
            "Employees must use the corporate VPN for all work activities. Core hours "
            "are 10 AM - 3 PM local time. Home office stipend: $500/year."
        ),
        metadata={"id": "DOC-001", "title": "Remote Work Policy"},
    ),
    Document(
        page_content=(
            "All software installations require IT approval via the ServiceNow catalog. "
            "Standard tools (Office 365, Slack, Zoom) are pre-approved. Non-standard "
            "software needs manager approval and security review (1-2 business days)."
        ),
        metadata={"id": "DOC-002", "title": "Software Request Process"},
    ),
    Document(
        page_content=(
            "Passwords must be at least 12 characters with uppercase, lowercase, number, "
            "and special character. Rotation every 90 days. Multi-factor authentication "
            "is mandatory for all cloud services. Report phishing via the Outlook button."
        ),
        metadata={"id": "DOC-003", "title": "Information Security Policy"},
    ),
    Document(
        page_content=(
            "New employees complete orientation on day one: HR paperwork, IT setup, "
            "office tour. Required training in first 30 days: cybersecurity awareness, "
            "code of conduct, anti-harassment, data privacy. Probation: 90 days."
        ),
        metadata={"id": "DOC-004", "title": "Employee Onboarding"},
    ),
    Document(
        page_content=(
            "Full-time employees receive 20 days PTO per year, accrued monthly. "
            "Sick leave: 10 days per year, no carryover. Parental leave: 12 weeks "
            "paid (primary), 4 weeks (secondary). PTO requests: 2 weeks notice for 3+ days."
        ),
        metadata={"id": "DOC-005", "title": "Leave and PTO Policy"},
    ),
]

# Chunk
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
)
chunks = splitter.split_documents(documents)
print(f"{len(documents)} documents → {len(chunks)} chunks")

# Embed and index  (Day 2: embeddings + vector store)
embeddings = HuggingFaceEmbeddings(model_name=CONFIG["embedding_model"])
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="capstone",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": CONFIG["retriever_top_k"]})

print(f"Indexed {vectorstore._collection.count()} chunks in Chroma")
print("[Day 2 component ✓] Vector store and retriever ready")

## Section B: Define the RAG chain (Day 3)

*This implements the **Day 3** component — retrieve context, then generate a grounded answer.*

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── YOUR SYSTEM PROMPT (adjust persona and rules for your use case) ──
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an internal company assistant. Answer the question using ONLY "
        "the provided context. If the answer is not in the context, say "
        "'I don't have that information in the current documents.' "
        "Cite source document IDs when possible. Be concise."
    )),
    ("user", "Context:\n{context}\n\nQuestion: {question}"),
])

parser = StrOutputParser()

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata.get('id', '?')}] {d.page_content}" for d in docs)

print("[Day 3 component ✓] RAG prompt template defined")

## Section C: Call the LLM API (Day 1)

*This implements the **Day 1** component — calling an external LLM API to generate text.*

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=CONFIG["llm_model"],
    temperature=CONFIG["temperature"],
    api_key=API_KEY or "sk-placeholder",
)

print(f"[Day 1 component ✓] LLM configured: {CONFIG['llm_model']}")

## Section D: End-to-end pipeline

Wire everything together: question → retrieve (Day 2) → prompt + LLM (Day 1 + Day 3) → answer with sources.

In [ ]:
# Full RAG chain (LCEL)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | parser
)

def ask(question):
    """Run the full pipeline and return answer + sources + latency."""
    start = time.perf_counter()

    # Retrieve (Day 2)
    docs = retriever.invoke(question)
    context = format_docs(docs)
    sources = [{"id": d.metadata.get("id"), "title": d.metadata.get("title")} for d in docs]

    # Generate (Day 1 + Day 3)
    if API_KEY:
        answer = (rag_prompt | llm | parser).invoke({
            "context": context,
            "question": question,
        })
    else:
        answer = "[Simulated] Grounded answer based on retrieved context."

    elapsed = time.perf_counter() - start

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "latency": round(elapsed, 3),
    }

print("Pipeline assembled — ready for queries.")

In [ ]:
# Demo: run a few questions through the pipeline
demo_questions = [
    "How do I request new software?",
    "What is the probation period for new employees?",
    "How many days of PTO do I get?",
]

simulated_answers = [
    "Submit requests through the ServiceNow catalog. Standard tools are pre-approved; non-standard software needs manager approval and security review. (DOC-002)",
    "The probation period for new employees is 90 days. (DOC-004)",
    "Full-time employees receive 20 days PTO per year, accrued monthly. (DOC-005)",
]

print("Demo queries:\n")
for i, q in enumerate(demo_questions):
    if API_KEY:
        result = ask(q)
    else:
        result = {
            "question": q,
            "answer": simulated_answers[i],
            "sources": [{"id": "DOC-???", "title": "Simulated"}],
            "latency": 0.5,
        }
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"Sources: {', '.join(s['id'] for s in result['sources'])}")
    print(f"Latency: {result['latency']}s\n")

## Responsible AI consideration (Day 3 Module 3)

Add at least one trust / safety measure to your assistant. Examples:

| Technique | Description |
|---|---|
| Source citations | Return document IDs so users can verify the answer |
| Out-of-scope refusal | Decline to answer questions not covered by documents |
| PII filtering | Redact personal information from outputs |
| Confidence threshold | Flag low-confidence answers for human review |
| Audit logging | Log queries, responses, and sources for accountability |

In [ ]:
import re

def pii_filter(text):
    """Redact email addresses and phone numbers from output."""
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.-]+", "[EMAIL REDACTED]", text)
    text = re.sub(r"\b\d{3}[-.]?\d{3}[-.]?\d{4}\b", "[PHONE REDACTED]", text)
    return text

def safe_ask(question):
    """Ask with responsible-AI guardrails."""
    result = ask(question)
    result["answer"] = pii_filter(result["answer"])
    return result

# Test
print("Responsible AI: PII filter active")
test = pii_filter("Contact alice@example.com or call 555-123-4567 for details.")
print(f"  Before: Contact alice@example.com or call 555-123-4567 for details.")
print(f"  After:  {test}")

## Evaluation (optional)

Check retrieval accuracy against known ground truth.

In [ ]:
ground_truth = [
    ("What VPN should I use?", "DOC-001"),
    ("How do I install VS Code?", "DOC-002"),
    ("What is the password policy?", "DOC-003"),
    ("What training do new employees need?", "DOC-004"),
    ("How much parental leave do I get?", "DOC-005"),
]

hits = 0
for q, expected in ground_truth:
    docs = retriever.invoke(q)
    ids = [d.metadata.get("id") for d in docs]
    found = expected in ids
    hits += found
    print(f"  {'HIT' if found else 'MISS':4s} | {q:45s} | Expected: {expected} | Got: {ids}")

recall = hits / len(ground_truth)
print(f"\nRecall@{CONFIG['retriever_top_k']}: {recall:.0%} ({hits}/{len(ground_truth)})")

## Your turn: customize the assistant

The scaffold above gives you a working baseline. Extend it for your chosen use case:

1. **Replace documents** — swap in your own domain data (Section A)
2. **Adjust the prompt** — change the persona and rules (Section B)
3. **Add features** — tools, memory, evaluation, guardrails
4. **Prepare the demo** — pick 2–3 compelling questions that highlight your design

In [ ]:
# ── YOUR CODE HERE ──
# Replace or extend any section above. Below are empty cells for experimentation.

# Example: add new documents
# new_docs = [Document(page_content="...", metadata={"id": "DOC-006", "title": "..."})]
# new_chunks = splitter.split_documents(new_docs)
# vectorstore.add_documents(new_chunks)
# print(f"Added {len(new_chunks)} chunks — total: {vectorstore._collection.count()}")

In [ ]:
# ── YOUR CODE HERE ──
# Example: build a more advanced chain with tools or memory

In [ ]:
# ── YOUR CODE HERE ──
# Example: evaluate your custom pipeline

## Presentation criteria

Each team presents for **5–10 minutes**. Cover these points:

| # | What to show | Details |
|---|---|---|
| 1 | **Use case** | One sentence: what does your assistant do? |
| 2 | **Architecture diagram** | Show which components map to Day 1, Day 2, Day 3 |
| 3 | **Live demo** | Run 1–3 questions through the pipeline; show answers + sources |
| 4 | **Responsible AI** | One sentence: how do you ensure trust? (e.g., "We cite sources so users can verify") |

**Tips:**
- Diagram + demo > slides
- Show real output, not screenshots
- Mention at least one limitation or future improvement

## Success criteria

- [x] Design uses an LLM API call (Day 1)
- [x] Design uses embeddings + vector store (Day 2)
- [x] Design uses RAG — retrieve + generate (Day 3)
- [x] Minimal working pipeline runs end-to-end
- [x] Presentation includes architecture diagram and live demo
- [x] At least one responsible-AI or evaluation point addressed

## Cleanup

In [ ]:
try:
    vectorstore._client.delete_collection("capstone")
    print("Temporary Chroma collection removed.")
except Exception:
    pass

print("Capstone complete.")